In [95]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the data into a DataFrame
df = pd.read_csv('test_dataset.csv')

# Define text preprocessing and TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')

# Extract features (text data) and target variable
X_text = df['review_text']
X_other = df.drop(columns=['review_text', 'Target'])  # Other features
y = df['Target']  # Target variable

# Split the data into training and testing sets
X_text_train, X_text_test, X_other_train, X_other_test, y_train, y_test = train_test_split(X_text, X_other, y, test_size=0.2, random_state=42)

# Preprocess and vectorize the text data
X_text_train_tfidf = tfidf_vectorizer.fit_transform(X_text_train)
X_text_test_tfidf = tfidf_vectorizer.transform(X_text_test)

# Combine text features with other features
import scipy.sparse as sp
X_train = sp.hstack([X_other_train.values, X_text_train_tfidf])
X_test = sp.hstack([X_other_test.values, X_text_test_tfidf])

# Train a Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7672752399129236
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.41      0.58     48226
           1       0.73      0.78      0.75    102268
           2       0.77      0.88      0.82    147635

    accuracy                           0.77    298129
   macro avg       0.83      0.69      0.72    298129
weighted avg       0.79      0.77      0.76    298129



In [96]:
from joblib import dump, load

# Save the trained Random Forest model
dump(model, 'random_forest_model_3.joblib')

['random_forest_model_3.joblib']

In [97]:
# Save the TF-IDF vectorizer
dump(tfidf_vectorizer, 'tfidf_vectorizer_3.joblib')

['tfidf_vectorizer_3.joblib']

In [1]:
from joblib import dump, load
import scipy.sparse as sp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the trained Random Forest model
model = load('random_forest_model_3.joblib')

# Load the TF-IDF vectorizer model (use the correct filename)
tfidf_vectorizer = load('tfidf_vectorizer_3.joblib')

# Load the new dataset into a DataFrame
new_df = pd.read_csv('test_rf_dataset.csv')

In [2]:
new_df

,review_text,Word_Count,Slang_Count,Character_Count,Punctuation_Count,Emoticon_Count,Emotion_Punctuation_Count,Capital_Letter_Count,Positive_Words_Count,Negative_Words_Count,Neutral_Words_Count,Target
0,I enjoy vintage books and movies so I enjoyed ...,57,0,240,6,0,0,5,3,1,49,1
1,This book is a reissue of an old one; the auth...,118,0,375,36,0,0,11,5,3,72,2
2,This was a fairly interesting read. It had ol...,72,0,311,15,0,0,7,6,1,55,2
3,I'd never read any of the Amy Brewster mysteri...,23,0,81,4,0,0,5,0,0,20,2
4,"If you like period pieces - clothing, lingo, y...",28,0,106,6,0,0,2,2,0,22,0
...,...,...,...,...,...,...,...,...,...,...,...,...
641166,Yasss hunny! This is a great read. That Dre is...,105,0,367,11,0,1,17,4,2,89,0
641167,I ENJOYED THIS BOOK FROM BEGINNING TO END NOW ...,58,0,216,2,0,0,213,3,5,48,2
641168,Great book! Cherika was a fool. She let that m...,73,0,236,9,0,4,9,3,3,58,2
641169,When I say this was an excellent book please b...,51,0,198,5,0,1,3,3,1,42,0


In [3]:
# Preprocess and vectorize the text data in the new dataset
X_text_new = new_df['review_text']
X_text_new_tfidf = tfidf_vectorizer.transform(X_text_new)

# Combine text features with other features in the new dataset
X_new = sp.hstack([new_df.drop(columns=['review_text', 'Target']).values, X_text_new_tfidf])

# Use the trained logistic regression model to predict the probabilities for the new dataset
y_pred_new = model.predict(X_new)

# Add the predicted target column to the new dataset
new_df['Predicted_Target'] = y_pred_new

# Optionally, save the new dataset with the predicted target column to a CSV file
new_df.to_csv('new_dataset_predicted.csv', index=False)

In [4]:
new_df

,review_text,Word_Count,Slang_Count,Character_Count,Punctuation_Count,Emoticon_Count,Emotion_Punctuation_Count,Capital_Letter_Count,Positive_Words_Count,Negative_Words_Count,Neutral_Words_Count,Target,Predicted_Target
0,I enjoy vintage books and movies so I enjoyed ...,57,0,240,6,0,0,5,3,1,49,1,1
1,This book is a reissue of an old one; the auth...,118,0,375,36,0,0,11,5,3,72,2,1
2,This was a fairly interesting read. It had ol...,72,0,311,15,0,0,7,6,1,55,2,1
3,I'd never read any of the Amy Brewster mysteri...,23,0,81,4,0,0,5,0,0,20,2,1
4,"If you like period pieces - clothing, lingo, y...",28,0,106,6,0,0,2,2,0,22,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
641166,Yasss hunny! This is a great read. That Dre is...,105,0,367,11,0,1,17,4,2,89,0,2
641167,I ENJOYED THIS BOOK FROM BEGINNING TO END NOW ...,58,0,216,2,0,0,213,3,5,48,2,2
641168,Great book! Cherika was a fool. She let that m...,73,0,236,9,0,4,9,3,3,58,2,1
641169,When I say this was an excellent book please b...,51,0,198,5,0,1,3,3,1,42,0,2


In [5]:
# Load the dataset into a DataFrame
temp_df = pd.read_csv('final_dataset_with_bert_vader_rf.csv')

temp_df.head()

,review_text,human_rating,BERT_Sentiment,VADER_Sentiment
0,I enjoy vintage books and movies so I enjoyed ...,5,4,5
1,This book is a reissue of an old one; the auth...,4,4,5
2,This was a fairly interesting read. It had ol...,4,4,5
3,I'd never read any of the Amy Brewster mysteri...,5,5,3
4,"If you like period pieces - clothing, lingo, y...",4,4,4


In [7]:
# Extract the needed columns from the dataset
needed_columns = temp_df[['human_rating', 'VADER_Sentiment', 'BERT_Sentiment']]

# Merge the columns with the 'new_df' DataFrame
new_df = pd.concat([new_df, needed_columns], axis=1)

# Check the number of rows and columns in the merged DataFrame
num_rows, num_columns = new_df.shape
print("Number of rows:", num_rows)
print("Number of columns:", num_columns)

# Now, the 'new_df' DataFrame contains the 'human_rating', 'VADER_Sentiment', and 'BERT_Sentiment' columns
new_df.tail()

Number of rows: 641171
Number of columns: 16


,review_text,Word_Count,Slang_Count,Character_Count,Punctuation_Count,Emoticon_Count,Emotion_Punctuation_Count,Capital_Letter_Count,Positive_Words_Count,Negative_Words_Count,Neutral_Words_Count,Target,Predicted_Target,human_rating,VADER_Sentiment,BERT_Sentiment
641166,Yasss hunny! This is a great read. That Dre is...,105,0,367,11,0,1,17,4,2,89,0,2,5,5,5
641167,I ENJOYED THIS BOOK FROM BEGINNING TO END NOW ...,58,0,216,2,0,0,213,3,5,48,2,2,5,4,5
641168,Great book! Cherika was a fool. She let that m...,73,0,236,9,0,4,9,3,3,58,2,1,5,2,5
641169,When I say this was an excellent book please b...,51,0,198,5,0,1,3,3,1,42,0,2,5,5,5
641170,This book was everything. I just hope Alexus w...,85,0,321,9,0,0,20,3,5,68,2,1,5,4,5


In [8]:
# Remove rows with target zero
#new_df = new_df[new_df['Target'] != 0]

In [9]:
new_df

,review_text,Word_Count,Slang_Count,Character_Count,Punctuation_Count,Emoticon_Count,Emotion_Punctuation_Count,Capital_Letter_Count,Positive_Words_Count,Negative_Words_Count,Neutral_Words_Count,Target,Predicted_Target,human_rating,VADER_Sentiment,BERT_Sentiment
0,I enjoy vintage books and movies so I enjoyed ...,57,0,240,6,0,0,5,3,1,49,1,1,5,5,4
1,This book is a reissue of an old one; the auth...,118,0,375,36,0,0,11,5,3,72,2,1,4,5,4
2,This was a fairly interesting read. It had ol...,72,0,311,15,0,0,7,6,1,55,2,1,4,5,4
3,I'd never read any of the Amy Brewster mysteri...,23,0,81,4,0,0,5,0,0,20,2,1,5,3,5
4,"If you like period pieces - clothing, lingo, y...",28,0,106,6,0,0,2,2,0,22,0,2,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
641166,Yasss hunny! This is a great read. That Dre is...,105,0,367,11,0,1,17,4,2,89,0,2,5,5,5
641167,I ENJOYED THIS BOOK FROM BEGINNING TO END NOW ...,58,0,216,2,0,0,213,3,5,48,2,2,5,4,5
641168,Great book! Cherika was a fool. She let that m...,73,0,236,9,0,4,9,3,3,58,2,1,5,2,5
641169,When I say this was an excellent book please b...,51,0,198,5,0,1,3,3,1,42,0,2,5,5,5


In [15]:
import numpy as np

# Create the 'final_sentiment' column based on the 'Predicted_Target' values
new_df['final_sentiment'] = np.where(new_df['Predicted_Target'] == 1, new_df['BERT_Sentiment'], new_df['VADER_Sentiment'])

# Now, the 'new_df' DataFrame contains the 'final_sentiment' column
new_df

,review_text,Word_Count,Slang_Count,Character_Count,Punctuation_Count,Emoticon_Count,Emotion_Punctuation_Count,Capital_Letter_Count,Positive_Words_Count,Negative_Words_Count,Neutral_Words_Count,Target,Predicted_Target,human_rating,VADER_Sentiment,BERT_Sentiment,final_sentiment
0,I enjoy vintage books and movies so I enjoyed ...,57,0,240,6,0,0,5,3,1,49,1,1,5,5,4,4
1,This book is a reissue of an old one; the auth...,118,0,375,36,0,0,11,5,3,72,2,1,4,5,4,4
2,This was a fairly interesting read. It had ol...,72,0,311,15,0,0,7,6,1,55,2,1,4,5,4,4
3,I'd never read any of the Amy Brewster mysteri...,23,0,81,4,0,0,5,0,0,20,2,1,5,3,5,5
4,"If you like period pieces - clothing, lingo, y...",28,0,106,6,0,0,2,2,0,22,0,2,4,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
641166,Yasss hunny! This is a great read. That Dre is...,105,0,367,11,0,1,17,4,2,89,0,2,5,5,5,5
641167,I ENJOYED THIS BOOK FROM BEGINNING TO END NOW ...,58,0,216,2,0,0,213,3,5,48,2,2,5,4,5,4
641168,Great book! Cherika was a fool. She let that m...,73,0,236,9,0,4,9,3,3,58,2,1,5,2,5,5
641169,When I say this was an excellent book please b...,51,0,198,5,0,1,3,3,1,42,0,2,5,5,5,5


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate MAE and MSE between human_rating and final_sentiment columns
mae_final_sentiment = mean_absolute_error(new_df['human_rating'], new_df['final_sentiment'])
mse_final_sentiment = mean_squared_error(new_df['human_rating'], new_df['final_sentiment'])

# Calculate MAE and MSE between human_rating and BERT_Sentiment columns
mae_bert_sentiment = mean_absolute_error(new_df['human_rating'], new_df['BERT_Sentiment'])
mse_bert_sentiment = mean_squared_error(new_df['human_rating'], new_df['BERT_Sentiment'])

# Calculate MAE and MSE between human_rating and VADER_Sentiment columns
mae_vader_sentiment = mean_absolute_error(new_df['human_rating'], new_df['VADER_Sentiment'])
mse_vader_sentiment = mean_squared_error(new_df['human_rating'], new_df['VADER_Sentiment'])

# Print the results
print("human_rating vs. final_sentiment:")
print("MAE:", mae_final_sentiment)
print("MSE:", mse_final_sentiment)
print()

print("human_rating vs. BERT_Sentiment:")
print("MAE:", mae_bert_sentiment)
print("MSE:", mse_bert_sentiment)
print()

print("human_rating vs. VADER_Sentiment:")
print("MAE:", mae_vader_sentiment)
print("MSE:", mse_vader_sentiment)

human_rating vs. final_sentiment:
MAE: 0.5026147470799521
MSE: 0.7972787290753949

human_rating vs. BERT_Sentiment:
MAE: 0.48123979406429795
MSE: 0.7295011783128058

human_rating vs. VADER_Sentiment:
MAE: 0.6605102227019001
MSE: 1.1540634245778427


In [17]:
from sklearn.metrics import classification_report

# Define true labels (y_true) and predicted labels for each approach
y_true = new_df['human_rating']
y_pred_hybrid = new_df['final_sentiment']
y_pred_bert = new_df['BERT_Sentiment']
y_pred_vader = new_df['VADER_Sentiment']

# Calculate classification report for each approach
report_hybrid = classification_report(y_true, y_pred_hybrid)
report_bert = classification_report(y_true, y_pred_bert)
report_vader = classification_report(y_true, y_pred_vader)

# Print the reports
print("Hybrid Approach:")
print(report_hybrid)

print("\nBERT Approach:")
print(report_bert)

print("\nVADER Approach:")
print(report_vader)

Hybrid Approach:
              precision    recall  f1-score   support

           1       0.43      0.43      0.43     16471
           2       0.30      0.46      0.36     22074
           3       0.40      0.36      0.38     60227
           4       0.36      0.34      0.35    152161
           5       0.75      0.76      0.76    390238

    accuracy                           0.61    641171
   macro avg       0.45      0.47      0.46    641171
weighted avg       0.60      0.61      0.60    641171


BERT Approach:
              precision    recall  f1-score   support

           1       0.42      0.48      0.45     16471
           2       0.31      0.55      0.40     22074
           3       0.41      0.51      0.45     60227
           4       0.40      0.57      0.47    152161
           5       0.86      0.64      0.74    390238

    accuracy                           0.61    641171
   macro avg       0.48      0.55      0.50    641171
weighted avg       0.68      0.61      0.63 

In [18]:
# Calculate Exact Match Accuracy for final sentiment
exact_match_accuracy_final = (new_df['final_sentiment'] == new_df['human_rating']).mean()

# Calculate Off-by-1 Accuracy for final sentiment
off_by_1_accuracy_final = ((new_df['final_sentiment'] - new_df['human_rating']).abs() <= 1).mean()

print("Final Sentiment:")
print("Exact Match Accuracy:", exact_match_accuracy_final)
print("Accuracy (Off-by-1):", off_by_1_accuracy_final)

# Calculate Exact Match Accuracy for BERT sentiment
exact_match_accuracy_bert = (new_df['BERT_Sentiment'] == new_df['human_rating']).mean()

# Calculate Off-by-1 Accuracy for BERT sentiment
off_by_1_accuracy_bert = ((new_df['BERT_Sentiment'] - new_df['human_rating']).abs() <= 1).mean()

print("\nBERT Sentiment:")
print("Exact Match Accuracy:", exact_match_accuracy_bert)
print("Accuracy (Off-by-1):", off_by_1_accuracy_bert)

# Calculate Exact Match Accuracy for VADER sentiment
exact_match_accuracy_vader = (new_df['VADER_Sentiment'] == new_df['human_rating']).mean()

# Calculate Off-by-1 Accuracy for VADER sentiment
off_by_1_accuracy_vader = ((new_df['VADER_Sentiment'] - new_df['human_rating']).abs() <= 1).mean()

print("\nVADER Sentiment:")
print("Exact Match Accuracy:", exact_match_accuracy_vader)
print("Accuracy (Off-by-1):", off_by_1_accuracy_vader)

Final Sentiment:
Exact Match Accuracy: 0.6051193831286817
Accuracy (Off-by-1): 0.9238814606399853

BERT Sentiment:
Exact Match Accuracy: 0.6059288395763377
Accuracy (Off-by-1): 0.94132454524612

VADER Sentiment:
Exact Match Accuracy: 0.5254729237598083
Accuracy (Off-by-1): 0.8657269277618607


In [19]:
# Count the occurrences of each value in the 'Predicted_Target' column
predicted_target_counts = new_df['Predicted_Target'].value_counts()

# Print the counts
print("Count of Predicted_Target values:")
print(predicted_target_counts)


Count of Predicted_Target values:
Predicted_Target
2    436253
1    202923
0      1995
Name: count, dtype: int64
